In [17]:
!pip install pandas numpy tensorflow

In [2]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, Concatenate, Dropout, LayerNormalization, MultiHeadAttention, Add, Reshape
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

print("--- 1. DEFINING FILE PATHS & LOADING DATA ---")
CLINICAL_CSV_PATH = 'raw_database/diagnosed_cbc_data_v4.csv'
GEO_CSV_PATH = 'raw_database/indiadata.csv'
IMAGE_DIR = 'raw_database'

df_clinical = pd.read_csv(CLINICAL_CSV_PATH)
df_geo = pd.read_csv(GEO_CSV_PATH)


print("\n--- 2. BUILDING CUSTOM IMAGE FEATURE EXTRACTOR ---")
image_input = Input(shape=(224, 224, 3), name="Raw_Image_Input")
x = Conv2D(32, (3, 3), activation='relu')(image_input)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(128, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)

# Project image features to a fixed 64-dimensional sequence space for the Transformer
image_feature_vector = Dense(64, activation='relu', name="Custom_Image_Vector")(x)
custom_feature_extractor = Model(inputs=image_input, outputs=image_feature_vector)
print("✅ Custom image feature extraction architecture built.")


print("\n--- 3. CLEANING AND VERIFYING DATASETS ---")
df_clinical['Target'] = df_clinical['Diagnosis'].apply(lambda x: 1 if 'anemia' in str(x).lower() else 0)
clinical_features = ['WBC', 'RBC', 'HGB', 'HCT', 'MCV', 'MCH', 'MCHC', 'PLT']

for col in clinical_features:
    df_clinical[col] = pd.to_numeric(df_clinical[col], errors='coerce')

df_clinical_clean = df_clinical.dropna(subset=clinical_features + ['Target']).reset_index(drop=True)
print(f"Cleaned Clinical Records ready for matrix assignment: {len(df_clinical_clean)} rows.")

df_geo['Geo_Risk_Score'] = (df_geo['AN_F'] + df_geo['AN_M']) / 200.0 
geo_risk_dict_clean = {str(k).strip().lower(): float(v) for k, v in zip(df_geo['STATE'], df_geo['Geo_Risk_Score']) if pd.notna(k)}
states_list = list(geo_risk_dict_clean.keys())


print("\n--- 4. SYNTHESIZING MULTIMODAL DATASET ---")
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
image_files = image_files[:200] 
print(f"Processing {len(image_files)} sample imagery entries...")

X_image_features = []
X_clinical_features = []
X_geo_features = []
y_targets = []

for img_name in image_files:
    img_path = os.path.join(IMAGE_DIR, img_name)
    try:
        img = image.load_img(img_path, target_size=(224, 224)) 
        img_array = image.img_to_array(img) / 255.0  
        img_array = np.expand_dims(img_array, axis=0)
        img_vector = custom_feature_extractor.predict(img_array, verbose=0)[0]
        
        rand_idx = random.randint(0, len(df_clinical_clean) - 1)
        clinical_row = df_clinical_clean.iloc[rand_idx]
        clin_vector = clinical_row[clinical_features].values.astype('float32')
        target_label = float(clinical_row['Target']) 
        
        rand_state = random.choice(states_list)
        geo_score = float(geo_risk_dict_clean[rand_state]) 
        
        X_image_features.append(img_vector)
        X_clinical_features.append(clin_vector)
        X_geo_features.append([geo_score]) 
        y_targets.append(target_label)
    except Exception as e:
        continue

X_img = np.array(X_image_features, dtype='float32')
X_clin = np.array(X_clinical_features, dtype='float32')
X_geo = np.array(X_geo_features, dtype='float32')
y = np.array(y_targets, dtype='float32')

X_img_train, X_img_test, X_clin_train, X_clin_test, X_geo_train, X_geo_test, y_train, y_test = train_test_split(
    X_img, X_clin, X_geo, y, test_size=0.2, random_state=42
)


print("\n--- 5. ASSEMBLING THE MULTIMODAL TECHNIQUE ---")
input_img = Input(shape=(64,), name="Image_Features_Input")
input_clin = Input(shape=(8,), name="Clinical_Features_Input")
input_geo = Input(shape=(1,), name="Geo_Features_Input")

# Project features to uniform 64-dimensional feature representations
proj_clin = Dense(64, activation='relu')(input_clin)
proj_geo = Dense(64, activation='relu')(input_geo)

# 🟢 FIX: Clean reshape formatting to allow explicit cross-attention sequences
token_img = Reshape((1, 64))(input_img)
token_clin = Reshape((1, 64))(proj_clin)
token_geo = Reshape((1, 64))(proj_geo)

# Concat along sequence dimension -> shape: (batch, 3, 64)
multimodal_sequence = Concatenate(axis=1)([token_img, token_clin, token_geo])

# TRUE TRANSFORMER MULTI-HEAD ATTENTION BLOCK
attention_output = MultiHeadAttention(num_heads=4, key_dim=64)(multimodal_sequence, multimodal_sequence)
attention_output = Dropout(0.1)(attention_output)

# Safe dimensional addition for the residual connection
x_fused = Add()([multimodal_sequence, attention_output])
x_fused = LayerNormalization()(x_fused)

# Flatten back out to 1D feature tensors safely
fused_vector = Flatten()(x_fused)

# Prediction layers
d1 = Dense(64, activation='relu')(fused_vector)
d1 = Dropout(0.2)(d1)
output = Dense(1, activation='sigmoid', name="Anemia_Prediction_Output")(d1)

fusion_model = Model(inputs=[input_img, input_clin, input_geo], outputs=output)
fusion_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("🔥 Transformer FusionNet built successfully with aligned dimensions!")


print("\n--- 6. TRAINING THE MULTIMODAL TRANSFORMER FUSION NETWORK ---")
history = fusion_model.fit(
    [X_img_train, X_clin_train, X_geo_train], y_train,
    validation_data=([X_img_test, X_clin_test, X_geo_test], y_test),
    epochs=15, 
    batch_size=16
)

print("\n🏆 PROCESS COMPLETE: Transformer FusionNet is successfully processing and training!")

--- 1. DEFINING FILE PATHS & LOADING DATA ---

--- 2. BUILDING CUSTOM IMAGE FEATURE EXTRACTOR ---
✅ Custom image feature extraction architecture built.

--- 3. CLEANING AND VERIFYING DATASETS ---
Cleaned Clinical Records ready for matrix assignment: 1281 rows.

--- 4. SYNTHESIZING MULTIMODAL DATASET ---
Processing 183 sample imagery entries...

--- 5. ASSEMBLING THE MULTIMODAL TECHNIQUE ---
🔥 Transformer FusionNet built successfully with aligned dimensions!

--- 6. TRAINING THE MULTIMODAL TRANSFORMER FUSION NETWORK ---
Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 136ms/step - accuracy: 0.5479 - loss: 0.9016 - val_accuracy: 0.7297 - val_loss: 0.6738
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6096 - loss: 0.6400 - val_accuracy: 0.7027 - val_loss: 0.6464
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6370 - loss: 0.6933 - val_accuracy: 0.7027 - val_loss: 0.6166
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5068 - loss: 0.7002 - val_a

In [ ]:
# Save both the feature extractor and the final fusion network
custom_feature_extractor.save('image_extractor.h5')
fusion_model.save('anemia_fusionnet.h5')
print("Models saved successfully for Streamlit deployment!")